# Drawdown-aware Objective Screen v1

Runtime card: `runtime/reward/card/drawdown_aware_objective_screen_v1`

Purpose: diagnose why the current optimization loop can find strict OOS candidates with `+30pp` excess and `<15%` MaxDD, but still fail saved walk-forward windows. This is a reward/model-selection diagnostic notebook, not a runtime promotion.

## Governance

- **Direction**: `reward` with model-selection evidence.
- **Role**: `diagnose`.
- **Changed layer**: `reward` and `selection`; no shared `src` change in this run.
- **Canonical baseline**: retained SOXL qlib harness and current paper/runtime model family.
- **Direct comparison baseline**: `risk_budget_residual_drawdown_guard_v1` and `predictive_drawdown_risk_gate_v1`, because both can pass the strict full-window target but fail walk-forward.
- **Target windows**: strict OOS plus saved `wf_2020`, `wf_2022`, `wf_2023_2024`.
- **Acceptance criteria**: identify whether any existing candidate satisfies strict `relative_return >= 30pp`, strict `max_dd <= 15%`, and walk-forward risk/underparticipation guards.
- **Expected trade-off**: penalizing path drawdown and rebound underparticipation should demote strict-only winners that are likely overfit.
- **Failure mode**: if all candidates fail walk-forward, this notebook should still produce a concrete next training-objective recommendation rather than another post-hoc cap.
- **Comparison-table row key**: `drawdown_aware_objective_screen_v1`.

## Historical Reflection

- **Closest success**: `risk_budget_residual_drawdown_guard_v1` reached the strict OOS target (`+30pp`, `<15%` MaxDD).
- **Closest failure**: `predictive_drawdown_risk_gate_v1` and the cash-only/residual guard family passed strict screens but were blocked by `wf_2020` underparticipation and `wf_2023_2024` drawdown.
- **Why this is not a duplicate**: this run does not add another overlay threshold. It audits the selection/reward objective itself across existing candidates to decide what the next training objective must optimize.


## External Basis

Primary sources used to constrain the experiment design:

- Qlib paper and repository: keep the work inside a reproducible AI quant research pipeline with separate data/model/strategy/execution components.
  - https://arxiv.org/abs/2009.11189
  - https://github.com/microsoft/qlib
- Deep Momentum Networks: trend estimation and position sizing should be learned/optimized together, with turnover/cost regularization, rather than only using a hand cap after inference.
  - https://arxiv.org/abs/1904.04912
- DeepTrader: market-condition embeddings and negative maximum-drawdown reward are a precedent for making drawdown risk part of the RL objective.
  - https://ojs.aaai.org/index.php/AAAI/article/view/16144
- Constrained Max Drawdown: maximum drawdown can be used as an explicit portfolio optimization constraint, not only as an after-the-fact metric.
  - https://arxiv.org/abs/2401.02601
- Expected Maximum Drawdown RRL: recurrent reinforcement learning with expected maximum drawdown supports putting drawdown risk directly into trading objective design.
  - https://doi.org/10.1016/j.eswa.2017.06.023

Design implication for this harness: strict full-window return/DD pass is only candidate discovery. A promotable reward/selection objective must include walk-forward drawdown and rebound underparticipation penalties before promotion.


In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


ROOT = find_repo_root(Path.cwd())
CARD_DIR = ROOT / "runtime/reward/card/drawdown_aware_objective_screen_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)

STRICT_RELATIVE_TARGET = 30.0
STRICT_MAX_DD_TARGET = 15.0
WF_RELATIVE_FLOORS = {
    "wf_2020": -10.0,      # avoid severe rebound/trend underparticipation
    "wf_2022": 30.0,       # bear-window defense should still beat buy-hold materially
    "wf_2023_2024": 30.0,  # later OOS should preserve positive excess
}

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print(f"card_dir={CARD_DIR.relative_to(ROOT)}")
print(f"targets: strict excess >= {STRICT_RELATIVE_TARGET}pp, strict MaxDD <= {STRICT_MAX_DD_TARGET}%")


card_dir=runtime/reward/card/drawdown_aware_objective_screen_v1
targets: strict excess >= 30.0pp, strict MaxDD <= 15.0%


## Candidate Loader

The loader intentionally consumes existing runtime artifacts instead of replaying actors. It covers:

- `selection_overfit_audit_v1` strict daily target and walk-forward screens.
- `predictive_drawdown_risk_gate_v1` classifier screen.
- Recent execution selection tables and walk-forward screens.
- The saved residual drawdown guard walk-forward comparison, attached back to the strict residual candidate.


In [2]:
def _num(value: Any, default: float = np.nan) -> float:
    try:
        if value is None or (isinstance(value, float) and math.isnan(value)):
            return default
        return float(value)
    except Exception:
        return default


def _bool(value: Any) -> bool:
    if isinstance(value, str):
        return value.lower() in {"true", "1", "yes", "y"}
    return bool(value)


def build_candidate_pool(root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, Any]] = []
    wf_rows: list[dict[str, Any]] = []

    strict_path = root / "runtime/model_selection/card/selection_overfit_audit_v1/strict_daily_target_screen.csv"
    wf_path = root / "runtime/model_selection/card/selection_overfit_audit_v1/walkforward_daily_screen.csv"
    if strict_path.exists():
        strict = pd.read_csv(strict_path)
        for _, r in strict.iterrows():
            daily_stem = Path(str(r["path"])).stem
            cid = f"strict_screen::{r['candidate']}::{daily_stem}"
            rows.append({
                "candidate_id": cid,
                "family": r["candidate"],
                "source": "selection_overfit_audit_v1",
                "candidate_key": daily_stem,
                "return_pct": _num(r.get("return")),
                "buyhold_return": _num(r.get("buyhold_return")),
                "relative_return": _num(r.get("relative_return")),
                "sharpe": _num(r.get("sharpe")),
                "max_dd": _num(r.get("max_dd")),
                "avg_pos": _num(r.get("avg_pos")),
                "trades": _num(r.get("trades")),
                "strict_target_pass_reported": _bool(r.get("target_pass")),
            })

    if wf_path.exists():
        wf = pd.read_csv(wf_path)
        for _, r in wf.iterrows():
            wf_rows.append({
                "source": "selection_overfit_audit_v1",
                "family": r["candidate"],
                "candidate_key": Path(str(r["path"])).stem,
                "candidate_id": None,
                "wf": r["wf"],
                "return_pct": _num(r.get("return")),
                "relative_return": _num(r.get("relative_return")),
                "max_dd": _num(r.get("max_dd")),
                "sharpe": _num(r.get("sharpe")),
                "avg_pos": _num(r.get("avg_pos")),
                "trades": _num(r.get("trades")),
            })

    pred_path = root / "runtime/model_selection/card/predictive_drawdown_risk_gate_v1/selection_table.csv"
    if pred_path.exists():
        pred = pd.read_csv(pred_path)
        for _, r in pred.iterrows():
            key = str(r["candidate_key"])
            cid = f"predictive_drawdown_risk_gate_v1::{key}"
            rows.append({
                "candidate_id": cid,
                "family": "predictive_drawdown_risk_gate_v1",
                "source": "predictive_drawdown_risk_gate_v1",
                "candidate_key": key,
                "return_pct": _num(r.get("return")),
                "buyhold_return": _num(r.get("buyhold_return")),
                "relative_return": _num(r.get("relative_return")),
                "sharpe": _num(r.get("sharpe")),
                "max_dd": _num(r.get("max_dd")),
                "avg_pos": _num(r.get("avg_pos")),
                "trades": _num(r.get("trades")),
                "strict_target_pass_reported": _bool(r.get("strict_target_pass")),
            })
            for wf_name in WF_RELATIVE_FLOORS:
                wf_rows.append({
                    "source": "predictive_drawdown_risk_gate_v1",
                    "family": "predictive_drawdown_risk_gate_v1",
                    "candidate_key": key,
                    "candidate_id": cid,
                    "wf": wf_name,
                    "return_pct": _num(r.get(f"return_{wf_name}")),
                    "relative_return": _num(r.get(f"relative_return_{wf_name}")),
                    "max_dd": _num(r.get(f"max_dd_{wf_name}")),
                    "sharpe": np.nan,
                    "avg_pos": np.nan,
                    "trades": np.nan,
                })

    for table_path in sorted((root / "runtime/execution/card").glob("*/selection_table.csv")):
        card = table_path.parent.name
        table = pd.read_csv(table_path)
        for row_index, r in table.iterrows():
            key = str(r.get("candidate_id", r.get("candidate_key", row_index)))
            cid = f"{card}::{key}"
            rel = _num(r.get("relative_return"))
            dd = _num(r.get("max_dd"))
            rows.append({
                "candidate_id": cid,
                "family": card,
                "source": "execution_selection_table",
                "candidate_key": key,
                "return_pct": _num(r.get("return")),
                "buyhold_return": _num(r.get("buy_and_hold_return", r.get("buyhold_return"))),
                "relative_return": rel,
                "sharpe": _num(r.get("sharpe")),
                "max_dd": dd,
                "avg_pos": _num(r.get("avg_pos")),
                "trades": _num(r.get("trades")),
                "strict_target_pass_reported": bool(rel >= STRICT_RELATIVE_TARGET and dd <= STRICT_MAX_DD_TARGET),
            })

        wf_screen_path = table_path.parent / "walkforward_screen.csv"
        if wf_screen_path.exists():
            wf_table = pd.read_csv(wf_screen_path)
            for row_index, r in wf_table.iterrows():
                key = str(r.get("candidate_id", r.get("candidate_key", row_index)))
                cid = f"{card}::{key}"
                wf_rows.append({
                    "source": "execution_walkforward_screen",
                    "family": card,
                    "candidate_key": key,
                    "candidate_id": cid,
                    "wf": r["wf"],
                    "return_pct": _num(r.get("return")),
                    "relative_return": _num(r.get("relative_return")),
                    "max_dd": _num(r.get("max_dd")),
                    "sharpe": _num(r.get("sharpe")),
                    "avg_pos": _num(r.get("avg_pos")),
                    "trades": _num(r.get("trades")),
                })

    base = pd.DataFrame(rows).drop_duplicates(["candidate_id"]).reset_index(drop=True)

    # Attach exact family-level walk-forward rows only when there is exactly one strict row for a family.
    strict_family_to_ids: dict[str, list[str]] = {}
    for _, r in base[base["source"] == "selection_overfit_audit_v1"].iterrows():
        strict_family_to_ids.setdefault(str(r["family"]), []).append(str(r["candidate_id"]))
    for wr in wf_rows:
        if wr["candidate_id"] is None:
            ids = strict_family_to_ids.get(str(wr["family"]), [])
            if len(ids) == 1:
                wr["candidate_id"] = ids[0]

    # The strict residual drawdown guard has a separate saved walk-forward validation card.
    comp_path = root / "runtime/execution/card/risk_budget_residual_drawdown_guard_walkforward_v1/walkforward_comparison.csv"
    residual_ids = base.loc[
        (base["source"] == "selection_overfit_audit_v1")
        & (base["family"] == "risk_budget_residual_drawdown_guard_v1"),
        "candidate_id",
    ].tolist()
    if comp_path.exists() and residual_ids:
        comp = pd.read_csv(comp_path)
        for _, r in comp.iterrows():
            wf_rows.append({
                "source": "risk_budget_residual_drawdown_guard_walkforward_v1",
                "family": "risk_budget_residual_drawdown_guard_v1",
                "candidate_key": "residual_guard_selected",
                "candidate_id": residual_ids[0],
                "wf": r["window"],
                "return_pct": np.nan,
                "relative_return": _num(r.get("residual_relative_return")),
                "max_dd": _num(r.get("residual_max_dd")),
                "sharpe": np.nan,
                "avg_pos": np.nan,
                "trades": np.nan,
            })

    wf_df = pd.DataFrame([r for r in wf_rows if r.get("candidate_id")])
    if not wf_df.empty:
        wf_df = (
            wf_df.sort_values(["candidate_id", "wf", "max_dd", "relative_return"], ascending=[True, True, True, False])
            .drop_duplicates(["candidate_id", "wf"], keep="first")
            .reset_index(drop=True)
        )
    return base, wf_df


base_candidates, wf_candidates = build_candidate_pool(ROOT)
print(f"base_candidates={base_candidates.shape}, wf_candidates={wf_candidates.shape}")
base_candidates.head(3)


base_candidates=(225, 12), wf_candidates=(606, 11)


,candidate_id,family,source,candidate_key,return_pct,buyhold_return,relative_return,sharpe,max_dd,avg_pos,trades,strict_target_pass_reported
0,strict_screen::risk_budget_residual_drawdown_g...,risk_budget_residual_drawdown_guard_v1,selection_overfit_audit_v1,risk_budget_residual_drawdown_guard_v1_selecte...,176.604683,146.146723,30.457960,0.971461,14.642461,0.160945,267.0,True
1,strict_screen::risk_budget_crash_risk_factor_v...,risk_budget_crash_risk_factor_v1,selection_overfit_audit_v1,risk_budget_crash_risk_factor_v1_selected_soxl...,190.745831,146.146723,44.599109,1.002457,15.484927,0.163622,243.0,False
2,strict_screen::risk_budget_downside_vol_target...,risk_budget_downside_vol_target_v1,selection_overfit_audit_v1,risk_budget_downside_vol_target_v1_best_combo_...,177.206488,146.146723,31.059765,0.959340,16.277862,0.163318,402.0,False


In [3]:
def add_objective_scores(base: pd.DataFrame, wf_df: pd.DataFrame) -> pd.DataFrame:
    wide = base.copy()
    for wf_name in WF_RELATIVE_FLOORS:
        sub = wf_df[wf_df["wf"] == wf_name].set_index("candidate_id") if not wf_df.empty else pd.DataFrame()
        if not sub.empty:
            wide[f"return_{wf_name}"] = wide["candidate_id"].map(sub["return_pct"])
            wide[f"relative_return_{wf_name}"] = wide["candidate_id"].map(sub["relative_return"])
            wide[f"max_dd_{wf_name}"] = wide["candidate_id"].map(sub["max_dd"])
            wide[f"avg_pos_{wf_name}"] = wide["candidate_id"].map(sub["avg_pos"])
            wide[f"trades_{wf_name}"] = wide["candidate_id"].map(sub["trades"])
        else:
            wide[f"return_{wf_name}"] = np.nan
            wide[f"relative_return_{wf_name}"] = np.nan
            wide[f"max_dd_{wf_name}"] = np.nan
            wide[f"avg_pos_{wf_name}"] = np.nan
            wide[f"trades_{wf_name}"] = np.nan

    computed = []
    for _, row in wide.iterrows():
        rel = _num(row.get("relative_return"), default=np.nan)
        dd = _num(row.get("max_dd"), default=np.nan)
        strict_rel_gap = max(0.0, STRICT_RELATIVE_TARGET - rel) if pd.notna(rel) else 100.0
        strict_dd_gap = max(0.0, dd - STRICT_MAX_DD_TARGET) if pd.notna(dd) else 100.0
        strict_loss = strict_rel_gap**2 + 4.0 * strict_dd_gap**2

        wf_loss = 0.0
        missing = 0
        worst_wf_dd_gap = 0.0
        worst_wf_rel_gap = 0.0
        wf_all_guard_pass = True
        wf_rel_values: list[float] = []

        for wf_name, rel_floor in WF_RELATIVE_FLOORS.items():
            wf_rel = _num(row.get(f"relative_return_{wf_name}"), default=np.nan)
            wf_dd = _num(row.get(f"max_dd_{wf_name}"), default=np.nan)
            if pd.isna(wf_rel) or pd.isna(wf_dd):
                missing += 1
                wf_all_guard_pass = False
                continue
            wf_rel_values.append(wf_rel)
            dd_gap = max(0.0, wf_dd - STRICT_MAX_DD_TARGET)
            rel_gap = max(0.0, rel_floor - wf_rel)
            wf_loss += 2.0 * dd_gap**2 + 0.25 * rel_gap**2
            worst_wf_dd_gap = max(worst_wf_dd_gap, dd_gap)
            worst_wf_rel_gap = max(worst_wf_rel_gap, rel_gap)
            if wf_dd > STRICT_MAX_DD_TARGET or wf_rel < rel_floor:
                wf_all_guard_pass = False

        missing_wf_penalty = missing * 1000.0
        wf_relative_std = float(np.std(wf_rel_values)) if len(wf_rel_values) >= 2 else np.nan
        overfit_penalty = max(0.0, wf_relative_std - 45.0) * 2.0 if pd.notna(wf_relative_std) else 0.0
        robust_score = strict_loss + wf_loss + missing_wf_penalty + overfit_penalty

        computed.append({
            "strict_relative_gap": strict_rel_gap,
            "strict_dd_gap": strict_dd_gap,
            "strict_loss": strict_loss,
            "wf_loss": wf_loss,
            "missing_wf_penalty": missing_wf_penalty,
            "wf_missing_windows": missing,
            "wf_relative_std": wf_relative_std,
            "overfit_penalty": overfit_penalty,
            "worst_wf_dd_gap": worst_wf_dd_gap,
            "worst_wf_relative_gap": worst_wf_rel_gap,
            "strict_target_pass": bool(strict_rel_gap == 0.0 and strict_dd_gap == 0.0),
            "wf_all_guard_pass": bool(wf_all_guard_pass),
            "full_objective_pass": bool(strict_rel_gap == 0.0 and strict_dd_gap == 0.0 and wf_all_guard_pass),
            "robust_score": robust_score,
        })

    scored = pd.concat([wide, pd.DataFrame(computed)], axis=1)
    scored["strict_rank"] = scored["strict_loss"].rank(method="min", ascending=True)
    scored["robust_rank"] = scored["robust_score"].rank(method="min", ascending=True)
    scored["rank_gap_strict_minus_robust"] = scored["strict_rank"] - scored["robust_rank"]

    def bucket(r: pd.Series) -> str:
        if r["full_objective_pass"]:
            return "promotable_by_objective"
        if r["strict_target_pass"] and not r["wf_all_guard_pass"]:
            return "strict_pass_wf_blocked"
        if r["strict_dd_gap"] > 0 and r["relative_return"] >= STRICT_RELATIVE_TARGET:
            return "return_high_drawdown_high"
        if r["strict_relative_gap"] > 0 and r["max_dd"] <= STRICT_MAX_DD_TARGET:
            return "risk_ok_return_short"
        return "misses_strict_target"

    scored["objective_bucket"] = scored.apply(bucket, axis=1)
    return scored


objective_table = add_objective_scores(base_candidates, wf_candidates)
print("candidate_pool", len(objective_table))
print("strict target pass", int(objective_table["strict_target_pass"].sum()))
print("walk-forward guard pass", int(objective_table["wf_all_guard_pass"].sum()))
print("full objective pass", int(objective_table["full_objective_pass"].sum()))
objective_table["objective_bucket"].value_counts()


candidate_pool 225
strict target pass 12
walk-forward guard pass 0
full objective pass 0


objective_bucket
return_high_drawdown_high    210
strict_pass_wf_blocked        12
misses_strict_target           3
Name: count, dtype: int64

In [4]:
DISPLAY_COLUMNS = [
    "candidate_id", "family", "source", "relative_return", "max_dd", "avg_pos",
    "strict_loss", "wf_loss", "robust_score", "objective_bucket",
    "relative_return_wf_2020", "max_dd_wf_2020",
    "relative_return_wf_2022", "max_dd_wf_2022",
    "relative_return_wf_2023_2024", "max_dd_wf_2023_2024",
]

top_strict = objective_table.sort_values(["strict_loss", "relative_return"], ascending=[True, False]).head(25)
top_robust = objective_table.sort_values(["robust_score", "strict_loss"], ascending=[True, True]).head(25)

print("Top strict-only candidates")
display(top_strict[DISPLAY_COLUMNS])

print("Top robust-objective candidates")
display(top_robust[DISPLAY_COLUMNS])


Top strict-only candidates


,candidate_id,family,source,relative_return,max_dd,avg_pos,strict_loss,wf_loss,robust_score,objective_bucket,relative_return_wf_2020,max_dd_wf_2020,relative_return_wf_2022,max_dd_wf_2022,relative_return_wf_2023_2024,max_dd_wf_2023_2024
40,predictive_drawdown_risk_gate_v1::h63_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,39.569762,14.702265,0.161633,0.000000,517.483446,537.331626,strict_pass_wf_blocked,-54.194588,18.102212,76.657302,10.970665,38.310638,17.229977
41,predictive_drawdown_risk_gate_v1::h63_loss0.18...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,38.552738,14.642461,0.161519,0.000000,517.483446,537.331626,strict_pass_wf_blocked,-54.194588,18.102212,76.657302,10.970665,38.310638,17.229977
44,predictive_drawdown_risk_gate_v1::h42_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,37.698379,14.821542,0.161570,0.000000,452.851485,467.860359,strict_pass_wf_blocked,-48.722047,20.933222,76.657302,10.970665,38.771572,16.948846
38,predictive_drawdown_risk_gate_v1::h42_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,37.167568,14.642461,0.161443,0.000000,545.258494,566.274505,strict_pass_wf_blocked,-55.486265,18.005230,76.657302,10.970665,38.310638,17.229977
45,predictive_drawdown_risk_gate_v1::h42_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,36.628108,14.642461,0.161475,0.000000,503.676397,520.776041,strict_pass_wf_blocked,-51.149705,20.933222,76.657302,10.970665,38.310638,17.229977
42,predictive_drawdown_risk_gate_v1::h63_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,36.518953,14.642461,0.161450,0.000000,517.483446,537.331626,strict_pass_wf_blocked,-54.194588,18.102212,76.657302,10.970665,38.310638,17.229977
35,predictive_drawdown_risk_gate_v1::h21_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,35.443239,14.621835,0.161365,0.000000,293.436680,302.983652,strict_pass_wf_blocked,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242
39,predictive_drawdown_risk_gate_v1::h21_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,34.121165,14.642461,0.161131,0.000000,477.123084,495.286236,strict_pass_wf_blocked,-52.328716,18.102212,76.657302,10.970665,38.310638,17.229977
37,predictive_drawdown_risk_gate_v1::h21_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,33.500893,14.642461,0.161159,0.000000,349.968174,362.274938,strict_pass_wf_blocked,-45.820393,18.102212,76.657302,10.970665,38.318647,17.229977
43,predictive_drawdown_risk_gate_v1::h21_loss0.18...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,33.202881,14.821542,0.161220,0.000000,384.817967,397.039204,strict_pass_wf_blocked,-45.726878,20.280621,76.657302,10.970665,38.310638,17.229977


Top robust-objective candidates


,candidate_id,family,source,relative_return,max_dd,avg_pos,strict_loss,wf_loss,robust_score,objective_bucket,relative_return_wf_2020,max_dd_wf_2020,relative_return_wf_2022,max_dd_wf_2022,relative_return_wf_2023_2024,max_dd_wf_2023_2024
35,predictive_drawdown_risk_gate_v1::h21_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,35.443239,14.621835,0.161365,0.000000,293.436680,302.983652,strict_pass_wf_blocked,-42.486967,18.533823,76.657302,10.970665,39.485655,16.518242
92,predictive_drawdown_risk_gate_v1::h42_loss0.15...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,38.659059,15.484927,0.162728,0.940616,325.709710,335.398411,return_high_drawdown_high,-41.477450,20.933222,76.657302,10.970665,40.022437,16.948846
36,predictive_drawdown_risk_gate_v1::h21_loss0.18...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,33.145001,14.642461,0.161184,0.000000,334.688696,346.219098,strict_pass_wf_blocked,-44.956868,18.102212,76.657302,10.970665,38.310638,17.229977
131,predictive_drawdown_risk_gate_v1::h42_loss0.12...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,41.671155,15.484927,0.162683,0.940616,341.931524,351.774977,return_high_drawdown_high,-41.977751,20.933222,76.657302,10.970665,38.524339,17.817902
59,predictive_drawdown_risk_gate_v1::h21_loss0.12...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,41.308072,15.406526,0.162736,0.661053,350.785277,362.230115,return_high_drawdown_high,-43.374593,20.933222,76.657302,10.970665,41.567265,15.978052
37,predictive_drawdown_risk_gate_v1::h21_loss0.22...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,33.500893,14.642461,0.161159,0.000000,349.968174,362.274938,strict_pass_wf_blocked,-45.820393,18.102212,76.657302,10.970665,38.318647,17.229977
74,predictive_drawdown_risk_gate_v1::h21_loss0.10...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,43.290567,15.406526,0.162136,0.661053,357.494011,368.531404,return_high_drawdown_high,-43.374593,20.933222,76.657302,10.970665,39.643397,17.076283
111,predictive_drawdown_risk_gate_v1::h21_loss0.12...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,41.814417,15.484927,0.162910,0.940616,357.494011,368.753414,return_high_drawdown_high,-43.374593,20.933222,76.657302,10.970665,39.359181,17.076283
77,predictive_drawdown_risk_gate_v1::h42_loss0.12...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,40.046190,15.406526,0.162207,0.661053,361.809752,372.736911,return_high_drawdown_high,-43.553415,20.933222,76.657302,10.970665,38.277302,17.229977
76,predictive_drawdown_risk_gate_v1::h42_loss0.10...,predictive_drawdown_risk_gate_v1,predictive_drawdown_risk_gate_v1,40.912093,15.406526,0.162230,0.661053,361.809752,372.778884,return_high_drawdown_high,-43.553415,20.933222,76.657302,10.970665,38.494190,17.229977


In [5]:
family_summary = (
    objective_table.groupby("family")
    .agg(
        candidates=("candidate_id", "count"),
        strict_pass=("strict_target_pass", "sum"),
        wf_guard_pass=("wf_all_guard_pass", "sum"),
        full_pass=("full_objective_pass", "sum"),
        best_relative_return=("relative_return", "max"),
        best_max_dd=("max_dd", "min"),
        best_robust_score=("robust_score", "min"),
        median_robust_score=("robust_score", "median"),
    )
    .reset_index()
    .sort_values(["full_pass", "strict_pass", "best_robust_score"], ascending=[False, False, True])
)

print("Family summary")
display(family_summary.head(30))


Family summary


,family,candidates,strict_pass,wf_guard_pass,full_pass,best_relative_return,best_max_dd,best_robust_score,median_robust_score
13,predictive_drawdown_risk_gate_v1,120,11,0,0,44.837918,14.621835,302.983652,435.884657
24,risk_budget_residual_drawdown_guard_v1,1,1,0,0,30.457960,14.642461,537.331626,537.331626
16,recovery_reentry_conditional_vol_target_v1,19,0,0,0,2367.698755,30.923275,1415.392939,1538.940744
14,recovery_reentry_2021_trendbreak_cap_v1,17,0,0,0,207.197964,30.436657,1754.321060,1754.321060
17,recovery_reentry_cppi_floor_v1,13,0,0,0,1594.473476,32.120180,1883.764837,1910.953403
29,soxl_soxx_efficiency_chop_guard_v1,17,0,0,0,595.207043,31.957679,2049.667473,2056.198420
20,recovery_reentry_regime_drawdown_guard_v1,9,0,0,0,1535.684114,32.120180,2051.580271,2051.580271
18,recovery_reentry_equity_drawdown_guard_v1,1,0,0,0,93.333364,34.365251,2301.975385,2301.975385
21,risk_budget_crash_risk_factor_v1,1,0,0,0,44.599109,15.484927,3000.940616,3000.940616
22,risk_budget_downside_vol_target_v1,2,0,0,0,31.333073,16.277862,3006.531723,3006.531723


In [6]:
strict_pass = objective_table[objective_table["strict_target_pass"]].copy()
strict_blocked = strict_pass[~strict_pass["wf_all_guard_pass"]].copy()
robust_best = objective_table.sort_values(["robust_score", "strict_loss"]).iloc[0]
strict_best = objective_table.sort_values(["strict_loss", "relative_return"], ascending=[True, False]).iloc[0]

summary = {
    "experiment": "drawdown_aware_objective_screen_v1",
    "decision": "diagnostic_no_promotion",
    "candidate_pool_size": int(len(objective_table)),
    "strict_target_pass_count": int(objective_table["strict_target_pass"].sum()),
    "wf_guard_pass_count": int(objective_table["wf_all_guard_pass"].sum()),
    "full_objective_pass_count": int(objective_table["full_objective_pass"].sum()),
    "strict_pass_wf_blocked_count": int(len(strict_blocked)),
    "best_strict_candidate": {
        "candidate_id": str(strict_best["candidate_id"]),
        "relative_return": float(strict_best["relative_return"]),
        "max_dd": float(strict_best["max_dd"]),
        "robust_score": float(strict_best["robust_score"]),
        "objective_bucket": str(strict_best["objective_bucket"]),
    },
    "best_robust_candidate": {
        "candidate_id": str(robust_best["candidate_id"]),
        "relative_return": float(robust_best["relative_return"]),
        "max_dd": float(robust_best["max_dd"]),
        "robust_score": float(robust_best["robust_score"]),
        "objective_bucket": str(robust_best["objective_bucket"]),
        "wf_2020_relative_return": float(robust_best.get("relative_return_wf_2020", np.nan)),
        "wf_2020_max_dd": float(robust_best.get("max_dd_wf_2020", np.nan)),
        "wf_2023_2024_relative_return": float(robust_best.get("relative_return_wf_2023_2024", np.nan)),
        "wf_2023_2024_max_dd": float(robust_best.get("max_dd_wf_2023_2024", np.nan)),
    },
    "objective_gaps": [
        "strict full-window pass is not sufficient: every strict-pass candidate is blocked by walk-forward risk or rebound underparticipation",
        "wf_2020 underparticipation is the dominant failure among strict-pass low-drawdown candidates",
        "wf_2023_2024 drawdown remains above 15% for the low-drawdown selector family",
        "high-return recovery/reentry candidates carry 30%+ strict MaxDD, so cap-only tightening cannot satisfy the requested target without destroying trend capture",
    ],
    "next_training_objective": {
        "recommendation": "train/select with a drawdown-aware objective before runtime overlay promotion",
        "terms": [
            "path drawdown penalty on future account equity, not only same-day return",
            "walk-forward weighted selection loss over wf_2020, wf_2022, wf_2023_2024",
            "rebound opportunity-cost penalty when high-volatility positive-momentum windows are underheld",
            "turnover/trade-count regularization so risk terms do not become churn",
        ],
        "do_not_repeat": [
            "another strict-only selector",
            "another cash-only overlay that does not change saved walk-forward order generation",
            "global drawdown cap tightening without regime/rebound state separation",
        ],
    },
    "artifacts": {
        "notebook": str((CARD_DIR / "drawdown_aware_objective_screen_v1.ipynb").relative_to(ROOT)),
        "objective_candidate_table": str((CARD_DIR / "objective_candidate_table.csv").relative_to(ROOT)),
        "objective_top_strict": str((CARD_DIR / "objective_top_strict.csv").relative_to(ROOT)),
        "objective_top_robust": str((CARD_DIR / "objective_top_robust.csv").relative_to(ROOT)),
        "objective_family_summary": str((CARD_DIR / "objective_family_summary.csv").relative_to(ROOT)),
        "training_objective_recommendations": str((CARD_DIR / "training_objective_recommendations.json").relative_to(ROOT)),
    },
}

objective_table.to_csv(CARD_DIR / "objective_candidate_table.csv", index=False)
top_strict.to_csv(CARD_DIR / "objective_top_strict.csv", index=False)
top_robust.to_csv(CARD_DIR / "objective_top_robust.csv", index=False)
family_summary.to_csv(CARD_DIR / "objective_family_summary.csv", index=False)
with (CARD_DIR / "training_objective_recommendations.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "experiment": "drawdown_aware_objective_screen_v1",
  "decision": "diagnostic_no_promotion",
  "candidate_pool_size": 225,
  "strict_target_pass_count": 12,
  "wf_guard_pass_count": 0,
  "full_objective_pass_count": 0,
  "strict_pass_wf_blocked_count": 12,
  "best_strict_candidate": {
    "candidate_id": "predictive_drawdown_risk_gate_v1::h63_loss0.22_q0.10",
    "relative_return": 39.56976243215786,
    "max_dd": 14.702264862158376,
    "robust_score": 537.3316258766885,
    "objective_bucket": "strict_pass_wf_blocked"
  },
  "best_robust_candidate": {
    "candidate_id": "predictive_drawdown_risk_gate_v1::h21_loss0.22_q0.20",
    "relative_return": 35.44323907104907,
    "max_dd": 14.621835439928663,
    "robust_score": 302.9836515093705,
    "objective_bucket": "strict_pass_wf_blocked",
    "wf_2020_relative_return": -42.48696705808648,
    "wf_2020_max_dd": 18.533822711730465,
    "wf_2023_2024_relative_return": 39.48565533949625,
    "wf_2023_2024_max_dd": 16.51824186431039
  

## Conclusion

This notebook is a diagnostic keep / no promotion unless `full_objective_pass_count > 0` after execution. The expected useful output is the gap analysis: whether the current harness is over-selecting strict full-window winners that fail saved walk-forward windows.

If no full objective candidate passes, the next experiment should move drawdown and rebound opportunity terms into training or checkpoint selection. It should not be another post-hoc cap-only overlay.
